In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np


In [2]:
##load the ANN trained model, scaler pickle file and onehot encoder
model = load_model('ann_model.h5')

#load encoder and scaler
with open('one_hot_encoder_geo.pkl', 'rb') as f:
    one_hot_encoder_geo = pickle.load(f)
    #changed the name as it was in video
with open('label_encode_gender.pkl', 'rb') as f:
    label_encoder_gender = pickle.load(f)

with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

In [3]:
#example input data for prediction
input_data = {
    'CreditScore': 600,
    'Geography' : 'France',
    'Gender' : 'Male',
    'Age' : 40,
    'Tenure' : 3,
    'Balance' : 60000,
    'NumOfProducts' : 2,
    'HasCrCard' : 1,
    'IsActiveMember' : 1,
    'EstimatedSalary' : 50000
}

In [ ]:
#one hot encoding
geo_endcoded = one_hot_encoder_geo.transform([[input_data['Geography']]]).toarray()
df_geo_encoded = pd.DataFrame(geo_endcoded,columns=one_hot_encoder_geo.get_feature_names_out(['Geography']))
df_geo_encoded

/Users/kshitoki/Documents/annclassification/venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [5]:
input_data_df = pd.DataFrame([input_data])
input_data_df
input_data_df= pd.concat([input_data_df.drop(columns=['Geography']),df_geo_encoded],axis=1)
input_data_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,Male,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [ ]:
# label encoding 
input_data_df['Gender'] = label_encoder_gender.transform(input_data_df['Gender'])
input_data_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [ ]:
#scalraing the input data
X = input_data_df
X_scaled = scaler.transform(X)

In [8]:
#prediction
prediction = model.predict(X_scaled)
print(prediction)

1/1 [==============================] - 0s 32ms/step
[[0.02386187]]


In [9]:
prediction_prob = prediction[0][0]
print(prediction_prob)

0.02386187


In [10]:
if prediction_prob > 0.5:
    print("The customer is likely to churn.")
else:
    print("The customer is not likely to churn.")

The customer is not likely to churn.
